# Nifty 50 — Volatility Regime Detection
## GARCH(1,1) + Hidden Markov Model

**Author:** Kirti Beniwal  
**Keywords:** Time series, GARCH, HMM, Volatility, Regime detection, Nifty 50

---

### What this notebook does
1. Downloads Nifty 50 daily closing prices (2010–2024) from Yahoo Finance  
2. Computes log returns and summary statistics  
3. Fits GARCH(1,1) with Student-t innovations to extract conditional volatility  
4. Fits a 2-state Gaussian HMM on the volatility series to label market regimes  
5. Visualises regimes on the price chart, return series, and volatility series  
6. Runs a simple regime-filtered trading strategy vs buy-and-hold  

### Mathematical background

**GARCH(1,1):**
$$r_t = \varepsilon_t, \quad \varepsilon_t = \sigma_t z_t, \quad z_t \sim t_\nu$$
$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

**HMM:**
- Hidden states: $S_t \in \{0\text{ (low vol)}, 1\text{ (high vol)}\}$  
- Transition matrix: $A_{ij} = P(S_t = j \mid S_{t-1} = i)$  
- Emissions: $\sigma_t \mid S_t = k \sim \mathcal{N}(\mu_k, \sigma^2_k)$  
- Decoding: Viterbi algorithm

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_nifty, get_summary_stats
from src.garch_model import select_garch_order, fit_garch, garch_diagnostics
from src.hmm_model   import fit_hmm, regime_statistics, select_hmm_states
from src.visualise   import (plot_full_dashboard, plot_vol_distributions,
                             plot_transition_matrix, plot_annual_regime_breakdown)
from src.backtesting import run_backtest, performance_metrics, plot_equity_curves

print('All imports OK')

## Step 1 — Load Nifty 50 data

In [ ]:
df = load_nifty(start='2010-01-01', end='2024-12-31')
stats = get_summary_stats(df)
df.tail()

In [ ]:
# Quick plot: price and raw returns
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(df.index, df['Close'], lw=0.8, color='#1a6ca8')
ax1.set_title('Nifty 50 closing price', fontsize=11)
ax1.set_ylabel('Index level')
ax2.plot(df.index, df['log_return']*100, lw=0.4, color='#555', alpha=0.8)
ax2.axhline(0, color='black', lw=0.5, linestyle='--')
ax2.set_title('Daily log returns (%)', fontsize=11)
ax2.set_ylabel('Log return (%)')
plt.tight_layout()
plt.show()

## Step 2 — Fit GARCH model

We grid-search GARCH(p,q) for p,q ∈ {1,2} using AIC, then fit the best order.

In [ ]:
best_p, best_q = select_garch_order(df['return_demeaned'], max_p=2, max_q=2)

In [ ]:
garch_result, cond_vol = fit_garch(df['return_demeaned'], p=best_p, q=best_q)
garch_diagnostics(garch_result, df['return_demeaned'])

In [ ]:
# Plot conditional volatility
plt.figure(figsize=(14, 4))
plt.plot(cond_vol.index, cond_vol*100, color='#c0392b', lw=0.9)
plt.axhline(20, color='gray', lw=0.8, linestyle='--', alpha=0.7, label='20% threshold')
plt.axhline(30, color='gray', lw=0.8, linestyle=':',  alpha=0.7, label='30% threshold')
plt.title('GARCH conditional volatility — annualised (%)', fontsize=11)
plt.ylabel('Ann. volatility (%)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Step 3 — Fit HMM and detect regimes

We first compare 2, 3, and 4 states by AIC, then fit the 2-state model.

In [ ]:
best_n = select_hmm_states(cond_vol, max_states=4)

In [ ]:
hmm_model, regime_series, scaler = fit_hmm(cond_vol, n_states=2, random_state=42)
stats_df = regime_statistics(df, regime_series)
stats_df

## Step 4 — Visualise

In [ ]:
plot_full_dashboard(df, cond_vol, regime_series)

In [ ]:
plot_vol_distributions(cond_vol, regime_series)

In [ ]:
low_state = int(np.argmin(hmm_model.means_.flatten()))
plot_transition_matrix(hmm_model, low_state)

In [ ]:
plot_annual_regime_breakdown(regime_series)

## Step 5 — Regime-filtered trading strategy vs Buy-and-hold

Strategy: Long Nifty when regime = Low vol (0); cash when regime = High vol (1).  
Position uses **previous day's regime** to avoid look-ahead bias.

In [ ]:
bt      = run_backtest(df, regime_series)
metrics = performance_metrics(bt)
metrics

In [ ]:
plot_equity_curves(bt)

---
## Summary of key findings

| Metric | Value |
|---|---|
| GARCH persistence (α+β) | ~0.97–0.99 (high — shocks last weeks) |
| Avg days in low-vol regime | ~40–60 trading days |
| Avg days in high-vol regime | ~15–25 trading days |
| Low-vol mean ann. volatility | ~10–14% p.a. |
| High-vol mean ann. volatility | ~25–35% p.a. |

**Interview one-liner:** *"GARCH extracts time-varying volatility from Nifty 50 log returns, capturing volatility clustering. A 2-state HMM then segments this volatility series into calm and stressed market regimes using the Viterbi algorithm, producing an interpretable daily regime label with practical risk management applications."*